# Capítulo 4 — Aprendizado por reforço sob ataque

## Missão
Um robô de armazém da Fortaleza Digital aprende por **reforço** a chegar à doca sem cair nos buracos. Os capítulos 5, 6 e 10 do livro mostram que um atacante pode:
- **perturbar as observações**, fazendo o agente achar que está noutro lugar;
- **envenenar as recompensas** durante o treino.

Você vai treinar o agente com Q-learning, atacar dos dois jeitos e defender com **redundância de sensores e voto da maioria**, a fusão de informação do cap. 8.

**No fim deste caderno você terá:** o módulo `gabarito_escudo/rl_seguro.py` (o mundo em grade, Q-learning, o ataque às observações, o envenenamento de recompensas e a defesa por voto) com a comparação com e sem defesa.

**Leitura essencial (opcional, livro em inglês):**
- 5.2 Characterizing Attacks on DRL Systems · PDF p. 93
- 5.4 Policy Induction Attacks · PDF p. 94
- 6.4 Attacks On DRL Systems · PDF p. 104
- 6.5 Defenses Against DRL System Attacks · PDF p. 107
- 8.7 Information Fusion Defense · PDF p. 139
- 10.4 Robustness On Adversarial State Observations · PDF p. 159

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_escudo/
os.makedirs("../gabarito_escudo", exist_ok=True)
open("../gabarito_escudo/__init__.py", "a").close()        # gabarito_escudo/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_escudo")

## Ideia em 30 segundos
O agente escolhe a ação pela **observação** do estado. Se o atacante troca a observação por um estado vizinho "ruim", a política faz a coisa errada, sem mudar uma vírgula do modelo.

**Defesa por fusão:** ler a posição por vários sensores independentes (GPS, odometria, câmera) e usar o voto da maioria. Se o atacante engana cada sensor com probabilidade `p`, enganar a maioria de 3 é bem mais raro: `3p²(1-p) + p³`.

### 4.1 **O mundo e o agente.** Crie `rl_seguro.py` com:
- `MAPA`: uma grade 6×6 de strings, com `S` a saída, `D` a doca (+10), `B` os buracos (-10, fim) e `.` o chão (-0,1 por passo);
- `passo(pos, acao, r)`: as ações são cima, baixo, esquerda e direita, com 10% de escorregar para uma ação aleatória;
- `treinar(episodios, envenenar=0.0, semente)`: Q-learning. Com `envenenar`, a recompensa ao chegar à doca vira -10 nessa fração de episódios;
- `avaliar(Q, episodios, ataque=0.0, sensores=1)`: cada sensor é enganado, com probabilidade `ataque`, para mostrar o vizinho **mais perigoso** (o de menor `max Q`). O agente usa a leitura mais votada entre os sensores. Devolve a taxa de sucesso e a de queda.

_A célula grava `gabarito_escudo/rl_seguro.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_escudo/rl_seguro.py
"""Aprendizado por reforço sob ataque: Q-learning num armazém, ataques às observações e às recompensas."""
from collections import Counter

import numpy as np

MAPA = ["S..B..",
        ".B...B",
        "...B..",
        ".B....",
        "...BB.",
        "B....D"]
ACOES = [(-1, 0), (1, 0), (0, -1), (0, 1)]
N = len(MAPA)


def _celula(p):
    return MAPA[p[0]][p[1]]


def passo(pos, acao, r):
    if r.random() < 0.1:
        acao = int(r.integers(4))
    d = ACOES[acao]
    nova = (min(max(pos[0] + d[0], 0), N - 1), min(max(pos[1] + d[1], 0), N - 1))
    c = _celula(nova)
    if c == "D":
        return nova, 10.0, True
    if c == "B":
        return nova, -10.0, True
    return nova, -0.1, False


def _vizinhos(p):
    return [(min(max(p[0] + a, 0), N - 1), min(max(p[1] + b, 0), N - 1)) for a, b in ACOES]


def _idx(p):
    return p[0] * N + p[1]


def treinar(episodios=8000, envenenar=0.0, alfa=0.1, gama=0.95, eps=0.2, semente=0):
    r = np.random.default_rng(semente)
    Q = np.zeros((N * N, 4))
    for _ in range(episodios):
        pos, fim, n = (0, 0), False, 0
        veneno = r.random() < envenenar
        while not fim and n < 100:
            a = int(r.integers(4)) if r.random() < eps else int(np.argmax(Q[_idx(pos)]))
            nova, rec, fim = passo(pos, a, r)
            if veneno and _celula(nova) == "D":
                rec = -10.0
            alvo = rec if fim else rec + gama * Q[_idx(nova)].max()
            Q[_idx(pos), a] += alfa * (alvo - Q[_idx(pos), a])
            pos, n = nova, n + 1
    return Q


def avaliar(Q, episodios=1000, ataque=0.0, sensores=1, semente=1):
    r = np.random.default_rng(semente)
    sucesso = queda = 0
    for _ in range(episodios):
        pos, fim, n = (0, 0), False, 0
        while not fim and n < 100:
            pior = min(_vizinhos(pos), key=lambda v: Q[_idx(v)].max())
            leituras = [pior if r.random() < ataque else pos for _ in range(sensores)]
            obs = Counter(leituras).most_common(1)[0][0]
            pos, rec, fim = passo(pos, int(np.argmax(Q[_idx(obs)])), r)
            n += 1
        sucesso += _celula(pos) == "D"
        queda += _celula(pos) == "B"
    return sucesso / episodios, queda / episodios

### 4.2 **Treine e avalie.** Treine o agente, mostre a política aprendida no mapa (setas `↑ ↓ ← →` na melhor ação de cada casa) e as taxas de sucesso e de queda sem ataque. Mesmo sem ataque, algumas quedas vêm dos 10% de escorregão.

In [ ]:
from gabarito_escudo.rl_seguro import MAPA, treinar, avaliar
Q = treinar()   # 8.000 episódios: com escorregões, o Q-learning precisa de taxa baixa e muita prática
setas = "↑↓←→"
for i, linha in enumerate(MAPA):
    print(" ".join(c if c in "BD" else setas[int(np.argmax(Q[i * 6 + j]))] for j, c in enumerate(linha)))
print("sem ataque (sucesso, queda):", avaliar(Q))

### 4.3 **Ataque às observações.** Avalie o agente com ataque em 10%, 20%, 30% e 50% dos passos. O atacante **não mexe** no ambiente nem no modelo, só no que o agente vê. Quanto a taxa de queda sobe?

In [ ]:
for at in (.1, .2, .3, .5):
    print(f"ataque {at:.0%}: (sucesso, queda) = {avaliar(Q, ataque=at)}")

### 4.4 **Envenenar as recompensas.** Treine com 20% e 40% dos episódios com a recompensa da doca envenenada (chegar vale -10). Com quanto veneno o agente passa a **evitar** o objetivo? Avalie sem ataque.

In [ ]:
for env in (.2, .4):
    print(f"recompensas envenenadas em {env:.0%}: (sucesso, queda) = {avaliar(treinar(envenenar=env))}")

### 4.5 **A defesa por fusão.** Com ataque de 20% e de 30% por sensor, compare 1, 3 e 5 sensores com voto da maioria. Calcule também a probabilidade teórica de a maioria de 3 ser enganada (`3p²(1-p) + p³`) e compare com o efeito observado.

In [ ]:
for p in (.2, .3):
    for s in (1, 3, 5):
        print(f"ataque {p:.0%}, {s} sensor(es): (sucesso, queda) = {avaliar(Q, ataque=p, sensores=s)}")
    print(f"   P(maioria de 3 enganada) = {3 * p ** 2 * (1 - p) + p ** 3:.3f}")

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
